# Raw ASGI

ASGI is the interface that Starlette and FastMCP sit on top of. An ASGI app is just:

```python
async def app(scope, receive, send)
```

- `scope` — dict with connection info (type, method, path, headers)
- `receive` — read the next event/chunk from the client
- `send` — write an event back to the client (always in order: start then body)

Two demo apps: one that only reads `scope`, one that reads the request body via `receive()`. `TestClient` runs the ASGI app in-process — no live server needed.

In [1]:
from IPython.display import display, Markdown

with open("01_raw_asgi.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
# Raw ASGI — the interface that Starlette and FastMCP sit on top of.
# An ASGI app is just:  async def app(scope, receive, send)
#   scope   — dict with connection info (type, method, path, headers)
#   receive — read the next event/chunk from the client
#   send    — write an event back to the client
#
# Run the demo:  python 01_raw_asgi.py
# Live server:   python -m uvicorn 01_raw_asgi:app --port 8080
# Requirements:  pip install starlette httpx uvicorn

from starlette.testclient import TestClient


# ── APP 1: ignores the body — reads scope only, never calls receive() ─────────
# Responds to any HTTP method and any path.
# The demo calls it with GET because POSTing to it is pointless — it never reads
# what you sent.

async def raw_app(scope, receive, send):
    if scope["type"] == "http":
        method = scope["method"]
        path   = scope["path"]

        await send({
            "type": "http.response.start",
            "status": 200,
            "headers": [[b"content-type", b"text/plain"]],
        })
        await send({
            "type": "http.response.body",
            "body": f"method={method} path={path}".encode(),
        })


# ── APP 2: reads the request body via receive(), branches on method ───────────
# Also responds to any path — there is no URL routing in raw ASGI.
# Method branching (GET vs POST) is done manually with scope["method"].

async def app(scope, receive, send):
    if scope["type"] != "http":
        return

    method = scope["method"]

    body_chunks = []
    while True:
        event = await receive()
        if event["type"] == "http.request":
            body_chunks.append(event.get("body", b""))
            if not event.get("more_body", False):
                break
        elif event["type"] == "http.disconnect":
            break

    request_body = b"".join(body_chunks)

    if method == "POST" and request_body:
        response_body = b"You posted: " + request_body
    else:
        response_body = b"Hello from raw ASGI - try POSTing some data"

    await send({
        "type": "http.response.start",
        "status": 200,
        "headers": [[b"content-type", b"text/plain"]],
    })
    await send({
        "type": "http.response.body",
        "body": response_body,
    })


# ── TestClient: runs the ASGI app in-process, no live server needed ───────────

if __name__ == "__main__":
    print("=== raw_app (receive not called) ===")
    client = TestClient(raw_app)
    r = client.get("/about")
    print("Status:", r.status_code)
    print("Body:  ", r.text)

    print()
    print("=== app — GET (empty body, hits fallback) ===")
    client2 = TestClient(app)
    r = client2.get("/")
    print("GET  Status:", r.status_code)
    print("GET  Body:  ", r.text)

    print()
    print("=== app — POST single chunk ===")
    r = client2.post("/", content=b"city=London&units=celsius")
    print("POST Status:", r.status_code)
    print("POST Body:  ", r.text)

    print()
    print("=== app — POST chunked body (exercises the receive loop) ===")
    def chunked_body():
        yield b"city=London"
        yield b"&units=celsius"

    r = TestClient(app).post("/", content=chunked_body())
    print(r.text)


# ── KEY MENTAL MODEL ──────────────────────────────────────────────────────────
#
# scope   — what kind of connection, and what does it want?
# receive — read the next event from the client
# send    — write an event back (always in order: start → body)
#
# HTTP = one request → one response (two send() calls)
# SSE  = one request → many sends held open  ← MCP over HTTP works this way
# WS   = bidirectional, scope["type"] == "websocket"
#
# FastMCP handles SSE for you — this file shows what it's doing underneath.

```

In [2]:
!python 01_raw_asgi.py

=== raw_app (receive not called) ===
Status: 200
Body:   method=GET path=/about

=== app — GET (empty body, hits fallback) ===
GET  Status: 200
GET  Body:   Hello from raw ASGI - try POSTing some data

=== app — POST single chunk ===
POST Status: 200
POST Body:   You posted: city=London&units=celsius

=== app — POST chunked body (exercises the receive loop) ===
You posted: city=London&units=celsius


# Starlette Routing

Starlette wraps the raw ASGI interface with a clean routing and request/response API. Instead of writing `scope`/`receive`/`send` by hand, you write async handler functions that receive a `Request` and return a `Response`.

- `Route("/path", handler)` — GET by default
- `request.query_params.get("key")` — read `?key=value` from the URL
- `await request.json()` — parse a JSON request body
- `JSONResponse(dict)` — serialise a dict to JSON and set Content-Type

FastMCP creates a Starlette app internally when you use HTTP/SSE transport.

In [3]:
from IPython.display import display, Markdown

with open("02_starlette_routing.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
# ─── STARLETTE ROUTING ───────────────────────────────────────────────────────
#
# Starlette wraps the raw ASGI interface in a clean routing and
# request/response API. Instead of writing scope/receive/send by hand,
# you write async handler functions that receive a Request and return a Response.
#
# FastMCP creates a Starlette app internally when you use HTTP/SSE transport.

from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse, PlainTextResponse
from starlette.routing import Route
from starlette.testclient import TestClient


# ─── ROUTE HANDLERS ──────────────────────────────────────────────────────────
# Each handler is an async function: Request -> Response.

async def homepage(request: Request) -> JSONResponse:
    return JSONResponse({"message": "MCP server running", "version": "1.0"})


async def health(request: Request) -> JSONResponse:
    return JSONResponse({"status": "ok"})


async def echo(request: Request) -> PlainTextResponse:
    # Read a query parameter from the URL: /echo?text=hello
    text = request.query_params.get("text", "nothing")
    return PlainTextResponse(f"You said: {text}")


async def create_item(request: Request) -> JSONResponse:
    # Read a JSON body from a POST request
    body = await request.json()
    name = body.get("name", "unnamed")
    return JSONResponse({"created": name}, status_code=201)


# ─── WIRE ROUTES TO HANDLERS ─────────────────────────────────────────────────

app = Starlette(routes=[
    Route("/",        homepage),
    Route("/health",  health),
    Route("/echo",    echo),
    Route("/items",   create_item, methods=["POST"]),
])


# ─── TEST WITHOUT RUNNING UVICORN ────────────────────────────────────────────
# Run with uvicorn: uvicorn 02_starlette_routing:app --reload

client = TestClient(app)

print(client.get("/").json())
print(client.get("/health").json())
print(client.get("/echo?text=MCP").text)
print(client.post("/items", json={"name": "weather_tool"}).json())


# ─── KEY MENTAL MODEL ────────────────────────────────────────────────────────
#
# Route("/path", handler)         — GET by default
# Route("/path", handler, methods=["POST"])  — restrict to specific HTTP methods
#
# request.query_params.get("key") — ?key=value in the URL
# await request.json()            — parse the JSON request body
# JSONResponse(dict)              — serialise dict to JSON and set Content-Type
#
# FastMCP transport options (verified against FastMCP source):
#   stdio (default)    — no Starlette, no routes, no web server
#   sse                — Route("/sse", methods=["GET"])  +  Mount("/messages/", ...)
#   streamable-http    — Route("/mcp", ...)  (no method restriction, default path /mcp)

```

In [4]:
!python 02_starlette_routing.py

{'message': 'MCP server running', 'version': '1.0'}
{'status': 'ok'}
You said: MCP
{'created': 'weather_tool'}


# Middleware

Middleware wraps every request before it reaches a route handler. First in the list = outermost wrapper (runs first in, last out).

Shows two middleware in one app:
- `CORSMiddleware` — handles cross-origin requests and preflight (`OPTIONS`) responses
- `LoggingMiddleware` — a custom class with `__call__(scope, receive, send)` that logs every request

FastMCP does not add `CORSMiddleware` — you add it yourself when mounting FastMCP into a larger app.

In [5]:
from IPython.display import display, Markdown

with open("03_middleware.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
# ─── MIDDLEWARE ──────────────────────────────────────────────────────────────
# Middleware wraps every request before it reaches a route handler.
# First in the list = outermost wrapper (runs first in, last out).
#
# FastMCP does not add CORSMiddleware — you add it yourself.

from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
from starlette.requests import Request
from starlette.responses import JSONResponse, Response
from starlette.routing import Route
from starlette.testclient import TestClient
from starlette.types import ASGIApp, Receive, Scope, Send


# ─── CUSTOM MIDDLEWARE ────────────────────────────────────────────────────────
# A middleware class wraps the next app in the chain.

class LoggingMiddleware:
    def __init__(self, app: ASGIApp):
        self.app = app

    async def __call__(self, scope: Scope, receive: Receive, send: Send):
        if scope["type"] == "http":
            path = scope.get("path", "")
            method = scope.get("method", "")
            print(f"[LOG] {method} {path}")
        await self.app(scope, receive, send)


# ─── APP WITH MIDDLEWARE ──────────────────────────────────────────────────────

async def homepage(request: Request) -> JSONResponse:
    return JSONResponse({"status": "ok"})


app = Starlette(
    routes=[Route("/", homepage)],
    middleware=[
        # CORS — allow any origin (use specific origins in production)
        Middleware(
            CORSMiddleware,
            allow_origins=["*"],
            allow_methods=["GET", "POST"],
            allow_headers=["*"],
        ),
        # Custom logging — runs on every request
        Middleware(LoggingMiddleware),
    ],
)


# ─── TEST ─────────────────────────────────────────────────────────────────────
# Run with uvicorn: uvicorn 03_middleware:app --reload

client = TestClient(app)

response = client.get("/")
print("Status:", response.status_code)

preflight = client.options(
    "/",
    headers={"Origin": "http://localhost:3000", "Access-Control-Request-Method": "POST"},
)
print("CORS preflight status:", preflight.status_code)
print("Allow-Origin header:", preflight.headers.get("access-control-allow-origin"))

```

In [6]:
!python 03_middleware.py

[LOG] GET /
Status: 200
CORS preflight status: 200
Allow-Origin header: *


# Mounting

`Mount` embeds one ASGI app inside another at a path prefix. It strips the prefix before forwarding to the sub-app — so the sub-app sees `/status`, not `/api/status`.

This is how you add a FastMCP server to a larger app without losing your own routes:

```python
app = Starlette(routes=[
    Route("/", my_homepage),
    Mount("/", app=mcp.sse_app()),
])
```

In [7]:
from IPython.display import display, Markdown

with open("04_mounting.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
# ─── MOUNTING ────────────────────────────────────────────────────────────────
# Mounting embeds one ASGI app inside another at a path prefix.
# This is how you add a FastMCP server to a larger app without losing your own routes:
#
#   app = Starlette(routes=[
#       Route("/", my_homepage),
#       Mount("/", app=mcp.sse_app()),  # Claude connects to /sse
#   ])

from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import PlainTextResponse, JSONResponse
from starlette.routing import Mount, Route
from starlette.testclient import TestClient


# ─── TWO SEPARATE SUB-APPS ────────────────────────────────────────────────────

async def api_status(request: Request) -> JSONResponse:
    return JSONResponse({"api": "running"})

async def api_version(request: Request) -> JSONResponse:
    return JSONResponse({"version": "1.0"})

api_app = Starlette(routes=[
    Route("/status",  api_status),
    Route("/version", api_version),
])


async def admin_dashboard(request: Request) -> PlainTextResponse:
    return PlainTextResponse("Admin dashboard")

admin_app = Starlette(routes=[
    Route("/", admin_dashboard),
])


# ─── COMBINED APP ─────────────────────────────────────────────────────────────

async def homepage(request: Request) -> PlainTextResponse:
    return PlainTextResponse("Main app — tools at /api, admin at /admin")


app = Starlette(routes=[
    Route("/",       homepage),
    Mount("/api",    app=api_app),     # api_app handles /api/status, /api/version
    Mount("/admin",  app=admin_app),   # admin_app handles /admin/
])


# ─── TEST ─────────────────────────────────────────────────────────────────────
# Run with uvicorn: uvicorn 04_mounting:app --reload

client = TestClient(app)

print(client.get("/").text)
print(client.get("/api/status").json())
print(client.get("/api/version").json())
print(client.get("/admin/").text)


# ─── WITH A REAL FASTMCP SERVER ───────────────────────────────────────────────
#
# from mcp.server.fastmcp import FastMCP
# import contextlib
# mcp = FastMCP("my_server")
#
# SSE (v1.x default — being superseded, no lifespan needed):
# app = Starlette(routes=[
#     Route("/", homepage),
#     Mount("/", app=mcp.sse_app()),   # Claude connects to /sse
# ])
#
# Streamable HTTP (current — requires lifespan):
# @contextlib.asynccontextmanager
# async def lifespan(app):
#     async with mcp.session_manager.run():
#         yield
#
# app = Starlette(
#     routes=[Route("/", homepage), Mount("/", app=mcp.streamable_http_app())],
#     lifespan=lifespan,               # Claude connects to /mcp
# )


# ─── KEY MENTAL MODEL ────────────────────────────────────────────────────────
#
# Mount("/prefix", app=sub_app)
#   — strips the prefix before forwarding to sub_app
#   — sub_app sees /status, not /api/status
#
# Route("/path", handler)
#   — exact match only (with optional path params)

```

In [8]:
!python 04_mounting.py

Main app — tools at /api, admin at /admin
{'api': 'running'}
{'version': '1.0'}
Admin dashboard


# Uvicorn

Uvicorn is the ASGI server — it binds to a port, accepts TCP connections, speaks HTTP, and calls your ASGI app for each request.

In the MCP stack: `uvicorn` → `Starlette` → `FastMCP` → your tools

Key CLI options:
- `--host` — default `127.0.0.1`; use `0.0.0.0` to accept external connections
- `--port` — default `8000`
- `--reload` — restart on file changes (dev only)

> **Note:** running this file starts a live server on port 8080 that blocks. Run it in a terminal instead:
> ```
> python 05_uvicorn_run.py
> ```

In [9]:
from IPython.display import display, Markdown

with open("05_uvicorn_run.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
# ─── UVICORN ─────────────────────────────────────────────────────────────────
# Uvicorn is an ASGI server — it binds to a port, accepts TCP connections,
# speaks HTTP, and calls your ASGI app for each request.
# In the MCP stack: [ uvicorn ] → [ Starlette ] → [ FastMCP ] → [ your tools ]
#
#
# ─── RUNNING FROM THE COMMAND LINE ───────────────────────────────────────────
#
# Basic:
#   uvicorn 05_uvicorn_run:app --port 8080
#
# Dev — auto-reload on file changes:
#   uvicorn 05_uvicorn_run:app --reload
#
# "05_uvicorn_run:app" means:
#   05_uvicorn_run  — the Python module (this file)
#   app             — the variable name of the ASGI app inside it
#
#
# ─── KEY OPTIONS ─────────────────────────────────────────────────────────────
#
#   --host    default 127.0.0.1   use 0.0.0.0 to accept external connections
#   --port    default 8000        TCP port to listen on
#   --reload  off                 restart on code changes (dev only)

import uvicorn
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route


# ─── THE APP ─────────────────────────────────────────────────────────────────

async def homepage(request: Request) -> JSONResponse:
    return JSONResponse({"status": "running", "transport": "http"})


app = Starlette(routes=[Route("/", homepage)])


# ─── RUN AND TEST ─────────────────────────────────────────────────────────────
# Terminal 1 — start the server:
#   python 05_uvicorn_run.py
#
# Terminal 2 — test it:
#   curl.exe http://localhost:8080/        (Windows — avoids PowerShell alias)
#   curl http://localhost:8080/            (Mac/Linux)
#   → {"status":"running","transport":"http"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080)


# ─── HOW FASTMCP USES UVICORN ────────────────────────────────────────────────
# host and port are set on the constructor, not on run().
#
# from mcp.server.fastmcp import FastMCP
# mcp = FastMCP("my_server", host="0.0.0.0", port=8080)
#
# if __name__ == "__main__":
#     mcp.run()                             # stdio (default) — no uvicorn
#     mcp.run(transport="sse")              # SSE — calls uvicorn.Config + uvicorn.Server
#     mcp.run(transport="streamable-http")  # streamable HTTP — same

```

# Full Stack — Pydantic + Starlette + Uvicorn

All three layers combined into the shape of a real MCP tool endpoint:

- **Pydantic** validates the incoming JSON request and serialises the response via `model_validate()` and `model_dump()`
- **Starlette** handles routing and `CORSMiddleware`
- **TestClient** runs the full stack in-process

The file also shows how FastMCP removes all of this boilerplate — Pydantic validation, Starlette routing, and uvicorn serving are all generated for you from a single `@mcp.tool()` decorated function.

In [10]:
from IPython.display import display, Markdown

with open("06_full_stack.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
# ─── FULL STACK — PYDANTIC + STARLETTE + UVICORN ────────────────────────────
#
# This file shows the complete HTTP stack in the shape of a real MCP tool
# endpoint: Pydantic validates data, Starlette handles routing, uvicorn serves.
#
# Run with: uvicorn 06_full_stack:app --reload
# Test with: curl -X POST http://localhost:8080/weather \
#              -H "Content-Type: application/json" \
#              -d '{"city": "London", "units": "celsius"}'

from pydantic import BaseModel, Field
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient


# ─── PYDANTIC MODELS ─────────────────────────────────────────────────────────

class WeatherRequest(BaseModel):
    city:  str = Field(description="City to get weather for")
    units: str = Field(default="celsius", description="celsius or fahrenheit")


class WeatherResponse(BaseModel):
    city:        str
    temperature: float
    units:       str
    condition:   str


# ─── TOOL LOGIC ───────────────────────────────────────────────────────────────

async def get_weather_logic(city: str, units: str) -> WeatherResponse:
    # In production: async HTTP call to a weather API
    temp = 65.3 if units == "fahrenheit" else 18.5
    return WeatherResponse(city=city, temperature=temp, units=units, condition="cloudy")


# ─── STARLETTE ROUTE HANDLERS ─────────────────────────────────────────────────

async def weather_endpoint(request: Request) -> JSONResponse:
    body = await request.json()

    # Pydantic validates the incoming JSON — returns 422 if invalid
    try:
        params = WeatherRequest.model_validate(body)
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=422)

    result = await get_weather_logic(params.city, params.units)

    # Pydantic serialises the response to a dict
    return JSONResponse(result.model_dump())


async def health(request: Request) -> JSONResponse:
    return JSONResponse({"status": "ok"})


# ─── STARLETTE APP ────────────────────────────────────────────────────────────

app = Starlette(
    routes=[
        Route("/weather", weather_endpoint, methods=["POST"]),
        Route("/health",  health),
    ],
    middleware=[
        Middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]),
    ],
)


# ─── TEST WITHOUT UVICORN ─────────────────────────────────────────────────────

if __name__ == "__main__":
    import json
    client = TestClient(app)

    # Valid request
    response = client.post("/weather", json={"city": "London", "units": "celsius"})
    print("Status:", response.status_code)
    print("Body:  ", json.dumps(response.json(), indent=2))

    # Missing required field — Pydantic catches it
    response2 = client.post("/weather", json={"units": "celsius"})
    print("\nMissing city — Status:", response2.status_code)
    print("Error:", response2.json()["error"][:80])

    # Health check
    print("\nHealth:", client.get("/health").json())

    print("\nTo run with uvicorn:")
    print("  uvicorn 06_full_stack:app --reload --port 8080")


# ─── HOW THIS MAPS TO AN ACTUAL FASTMCP SERVER ───────────────────────────────
#
# In FastMCP you don't write any of the Starlette or routing code by hand.
# FastMCP generates it for you from your tool functions:
#
#   from mcp.server.fastmcp import FastMCP
#
#   mcp = FastMCP("weather_server", host="0.0.0.0", port=8080)
#
#   @mcp.tool()
#   async def get_weather(city: str, units: str = "celsius") -> WeatherResponse:
#       return await get_weather_logic(city, units)
#
#   if __name__ == "__main__":
#       mcp.run(transport="sse")
#
# FastMCP does:
#   1. Uses Pydantic to validate inputs before calling your function
#   2. Creates a Starlette app with /sse and /messages/ routes
#   3. Serves it with uvicorn (via uvicorn.Config + uvicorn.Server)

```

In [11]:
!python 06_full_stack.py

Status: 200
Body:   {
  "city": "London",
  "temperature": 18.5,
  "units": "celsius",
  "condition": "cloudy"
}

Missing city — Status: 422
Error: 1 validation error for WeatherRequest
city
  Field required [type=missing, input

Health: {'status': 'ok'}

To run with uvicorn:
  uvicorn 06_full_stack:app --reload --port 8080
